# PCA Variance Inference on IMU Data

Legacy sections have been removed. This notebook keeps only a modern, reproducible PCA workflow for `imu_data.csv`.

## 1. Set Up Environment and Dependencies

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path("imu_data.csv")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 200)
print("Data path:", DATA_PATH)
print("Seed:", SEED)

## 2. Load Configuration and Input Data

In [ ]:
config = {
    "possible_targets": ["class", "label", "target", "y", "activity"],
    "variance_thresholds": [0.90, 0.95, 0.99],
    "test_size": 0.25,
}

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing input data: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## 3. Validate and Clean Data

In [ ]:
validation = {
    "rows": int(df.shape[0]),
    "cols": int(df.shape[1]),
    "null_count": int(df.isna().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
}

target_col = next((c for c in config["possible_targets"] if c in df.columns), df.columns[-1])

clean_df = df.copy()
for c in clean_df.columns:
    if c != target_col:
        clean_df[c] = pd.to_numeric(clean_df[c], errors="coerce")

clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("Validation summary:", validation)
print("Target column:", target_col)
clean_df.head()

## 4. Implement Modern Processing Pipeline

In [ ]:
feature_df = clean_df.drop(columns=[target_col], errors="ignore")
feature_df = feature_df.select_dtypes(include=[np.number])

if feature_df.shape[1] == 0:
    raise ValueError("No numeric feature columns available for PCA.")

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_imputed = imputer.fit_transform(feature_df)
X_scaled = scaler.fit_transform(X_imputed)

pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)
explained_ratio = pca_full.explained_variance_ratio_
cumulative_ratio = np.cumsum(explained_ratio)

plt.figure(figsize=(10, 5))
plt.bar(range(1, len(explained_ratio) + 1), explained_ratio, alpha=0.6, label="Individual")
plt.step(range(1, len(cumulative_ratio) + 1), cumulative_ratio, where="mid", color="red", label="Cumulative")
plt.axhline(0.95, color="green", linestyle="--", label="95% threshold")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("PCA Explained Variance")
plt.legend()
plt.tight_layout()
plt.show()

for threshold in config["variance_thresholds"]:
    k = int(np.argmax(cumulative_ratio >= threshold) + 1)
    print(f"Components needed for {int(threshold*100)}% variance: {k} (retained={cumulative_ratio[k-1]:.4f})")

n_components_95 = int(np.argmax(cumulative_ratio >= 0.95) + 1)
pca_95 = PCA(n_components=n_components_95)
X_pca_95 = pca_95.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca_95, columns=[f"PC{i}" for i in range(1, n_components_95 + 1)])
pca_df[target_col] = clean_df[target_col].values
pca_df.head()

## 5. Train and Tune Model

In [ ]:
can_train = not pca_df[target_col].isna().any()
if can_train:
    y = pca_df[target_col]
    X = pca_df.drop(columns=[target_col])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config["test_size"], random_state=SEED, stratify=y
    )

    baseline = SVC(kernel="rbf", C=1.0)
    baseline.fit(X_train, y_train)
    baseline_pred = baseline.predict(X_test)

    tuned = GridSearchCV(
        SVC(),
        param_grid={"C": [0.5, 1, 5], "kernel": ["linear", "rbf"]},
        cv=5,
        n_jobs=-1,
    )
    tuned.fit(X_train, y_train)
    tuned_pred = tuned.predict(X_test)

    train_outputs = {
        "baseline_acc": float(accuracy_score(y_test, baseline_pred)),
        "tuned_acc": float(accuracy_score(y_test, tuned_pred)),
        "best_params": tuned.best_params_,
    }
    print(train_outputs)
else:
    print("Target contains null values. Skipping supervised training.")
    train_outputs = None

## 6. Evaluate with Automated Metrics

In [ ]:
if train_outputs is not None:
    eval_summary = {
        "baseline_accuracy": train_outputs["baseline_acc"],
        "tuned_accuracy": train_outputs["tuned_acc"],
        "improvement": train_outputs["tuned_acc"] - train_outputs["baseline_acc"],
    }
    print(eval_summary)

    assert eval_summary["baseline_accuracy"] >= 0.0
    assert eval_summary["tuned_accuracy"] >= 0.0
    assert n_components_95 <= feature_df.shape[1]
else:
    eval_summary = {"status": "training_skipped"}
    print(eval_summary)

## 7. Persist Artifacts and Reproducible Outputs

In [ ]:
pca_df.to_csv(ARTIFACT_DIR / "pca_reduced_data.csv", index=False)

run_metadata = {
    "timestamp_utc": datetime.utcnow().isoformat(),
    "input_path": str(DATA_PATH),
    "rows": int(clean_df.shape[0]),
    "original_feature_count": int(feature_df.shape[1]),
    "components_95": int(n_components_95),
    "variance_thresholds": config["variance_thresholds"],
    "train_outputs": train_outputs,
    "eval_summary": eval_summary,
}

with open(ARTIFACT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)

print("Saved:")
print("-", ARTIFACT_DIR / "pca_reduced_data.csv")
print("-", ARTIFACT_DIR / "run_metadata.json")
run_metadata